### 1. Librerias

In [7]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
pd.options.mode.chained_assignment = None
from catboost import CatBoostClassifier

In [8]:
import sbp_modelo as sbp

In [9]:
PROCESSED_DATA_DIR = "../data/processed/"
MLRUNS_DIR = "../mlruns/"
LOGS_DIR = "../logs/"
MODELS_DIR = "../models/"
OPTUNA_DIR = "../optuna/"

In [10]:
cols_cat = ['type']
cols_num = ['amount','oldbalanceOrg','origTrxAmount','oldbalanceDest','destTrxAmount']
cols_total = cols_cat + cols_num

### 2. Carga modelo y data prod

In [11]:
df = pd.read_parquet(PROCESSED_DATA_DIR+"processed.parquet")

In [12]:
df_produccion = df[df['base'] == 2]
df_produccion.head()

,base,type,amount,entTrxType,oldbalanceOrg,origTrxAmount,oldbalanceDest,destTrxAmount,isFraud
5787030,2,CASH_OUT,58780.03,CC,30916.0,-30916.00,669365.85,58780.02,0
5787031,2,PAYMENT,11387.52,CM,19352.0,-11387.52,0.00,NaN,0
5787032,2,PAYMENT,3606.19,CM,11934.0,-3606.19,0.00,NaN,0
5787033,2,CASH_OUT,211759.90,CC,11099.0,-11099.00,0.00,211759.90,0
5787034,2,CASH_OUT,545893.11,CC,48663.0,-48663.00,7817994.67,545893.12,0


In [17]:
df_produccion = df_produccion.fillna(0)

In [18]:
import joblib
modelos = ['f1prauc_best_model.pkl', 'prauc_best_model.pkl', 'f1_best_model.pkl']
metricas = {}
for i, modelo in enumerate(modelos):
    model = joblib.load(MODELS_DIR+modelo)
    df_produccion['pred'+str(i)] = model.predict(df_produccion[cols_total])
    metricas[modelo] = sbp.metricas_clasificacion_binaria(df_produccion['isFraud'], df_produccion['pred'+str(i)], umbral=0.5)

In [15]:
metricas

{'f1prauc_best_model.pkl': {'AUC': 0.7039464940720056,
  'PR_AUC': 0.41145731673108765,
  'Accuracy': 0.9961673309264774,
  'F1': 0.5793287566742944,
  'Precision': 0.9993421052631579,
  'Recall': 0.40789473684210525,
  'Gini': 0.4078929881440112,
  'KS': 0.407892988144011},
 'prauc_best_model.pkl': {'AUC': 0.6779001074113856,
  'PR_AUC': 0.3599681990056274,
  'Accuracy': 0.9958320158171438,
  'F1': 0.5248564072093483,
  'Precision': 1.0,
  'Recall': 0.3558002148227712,
  'Gini': 0.35580021482277124,
  'KS': 0.3558002148227712},
 'f1_best_model.pkl': {'AUC': 0.7493190610004385,
  'PR_AUC': 0.4989646344523134,
  'Accuracy': 0.9967371928739458,
  'F1': 0.6641630901287554,
  'Precision': 0.9941113490364025,
  'Recall': 0.4986573576799141,
  'Gini': 0.4986381220008771,
  'KS': 0.49863812200087715}}

In [ ]:
# 'f1_best_model.pkl' es el mejor modelo, se puede optimizar con mas trials pero se necesita mas tiempo.